In [1]:
from typing import TypedDict, Optional, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from dotenv import load_dotenv
from datetime import date
from openai import OpenAI
import tempfile

openai_client = OpenAI()

load_dotenv()

True

In [2]:
SAFETY_REPLY = (
    "I'm really glad you shared that with me. "
    "You are not alone. Please call 988 or go to your nearest emergency room. "
    "A member of our care team has been notified. "
    "Whenever you're ready, we can continue with the intake."
)

REQUIRED_SECTIONS = {
    "general_information",
    "current_concerns",
    "past_psychiatric_history",
    "medications",
    "medical_history",
    "medication_allergies",
    "family_psychiatric_history",
    "substance_use",
    "education_employment_legal_history",
    "personal_history",
    "phq9",
    "gad7",
}

In [3]:
class GeneralInformation(TypedDict):
    name: Optional[str]
    gender: Optional[str]
    date_of_birth: Optional[str]
    address: Optional[str]
    postal_code: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    has_family_doctor: Optional[bool]
    family_doctor_name: Optional[str]
    has_psychiatrist: Optional[bool]
    psychiatrist_name: Optional[str]

class CurrentConcerns(TypedDict):
    primary_problems: Optional[str]
    goals: Optional[str]

class PastPsychiatricHistory(TypedDict):
    previously_assessed:   Optional[bool]
    assessment_details:    Optional[str]
    previous_diagnoses:    Optional[str]
    hospital_admission:    Optional[bool]
    hospital_details:      Optional[str]
    self_harm_history:     Optional[bool]
    self_harm_details:     Optional[str]
    previous_therapies:    Optional[str]
    previous_treatments:   Optional[str]

class Medications(TypedDict):
    has_current_medications: Optional[bool]
    current_medications: Optional[str]
    has_past_medications: Optional[bool]
    past_medication_details: Optional[str]

class MedicalHistory(TypedDict):
    has_medical_conditions: Optional[bool]
    medical_conditions: Optional[str]
    has_surgical_history: Optional[bool]
    surgical_history: Optional[str]
    height: Optional[str]
    weight: Optional[str]

class MedicationAllergies(TypedDict):
    has_medication_allergies: Optional[bool]
    medication_allergies: Optional[str]

class FamilyPsychiatricHistory(TypedDict):
    has_family_psychiatric_history: Optional[bool]
    family_psychiatric_history_details: Optional[str]

class SubstanceUse(TypedDict):
    alcohol_use: Optional[str]
    nicotine_use: Optional[str]
    marijuana_use: Optional[str]
    stimulant_use: Optional[str]
    opioid_use: Optional[str]
    benzodiazepine_use: Optional[str]
    hallucinogen_use: Optional[str]
    other_substance_use: Optional[str]

class EducationEmploymentLegalHistory(TypedDict):
    active_legal_cases: Optional[bool]
    legal_case_details: Optional[str]
    highest_education: Optional[str]
    employment_history: Optional[str]

class PersonalHistory(TypedDict):
    early_childhood: Optional[str]
    adolescence_early_adulthood: Optional[str]
    significant_relationships: Optional[str]
    future_goals: Optional[str]

class PHQ9(TypedDict):
    phq9_q1: Optional[int]
    phq9_q2: Optional[int]
    phq9_q3: Optional[int]
    phq9_q4: Optional[int]
    phq9_q5: Optional[int]
    phq9_q6: Optional[int]
    phq9_q7: Optional[int]
    phq9_q8: Optional[int]
    phq9_q9: Optional[int]
    phq9_difficulty: Optional[str]
    phq9_total: Optional[int]

class GAD7(TypedDict):
    gad7_q1: Optional[int]
    gad7_q2: Optional[int]
    gad7_q3: Optional[int]
    gad7_q4: Optional[int]
    gad7_q5: Optional[int]
    gad7_q6: Optional[int]
    gad7_q7: Optional[int]
    gad7_difficulty: Optional[str]
    gad7_total: Optional[int]

class AliceState(TypedDict):
    messages: List[dict]
    current_section: Optional[str]
    sections_completed: List[str]
    safety_flag: bool
    conversation_started: bool
    general_information: GeneralInformation
    current_concerns: CurrentConcerns
    past_psychiatric_history: PastPsychiatricHistory
    medications: Medications
    medical_history: MedicalHistory
    medication_allergies: MedicationAllergies
    family_psychiatric_history: FamilyPsychiatricHistory
    substance_use: SubstanceUse
    education_employment_legal_history: EducationEmploymentLegalHistory
    personal_history: PersonalHistory
    phq9: PHQ9    
    gad7: GAD7


llm = ChatOpenAI(model="gpt-5.4-mini")



In [4]:
@tool
def save_answer(field_name: str, value: str) -> str:
    """
    Call this every time the patient answers a question or corrects a previous answer.
    field_name: the exact field name (e.g. 'name', 'gender', 'age', 'address')
    value: exactly what the patient said
    """
    return "saved"


In [5]:
@tool
def raise_safety_flag() -> str:
    """
    Call this immediately if the patient mentions:
    - Thoughts of suicide or self-harm
    - Thoughts of harming others
    - A current crisis situation
    """
    return "flag_raised"



In [6]:
tools = [save_answer, raise_safety_flag]
model    = ChatOpenAI(model="gpt-5.4-mini").bind_tools(tools)



In [7]:
SUPPORTED_TEST_LANGUAGES = ["English", "French", "Hindi"]

STATIC_PROMPT = f"""
You are ALICE, a warm and professional clinical intake assistant
at St. Michael's Hospital, Unity Health Toronto.

Your role:
- Help collect clinical intake information before the patient's appointment.
- Collect information one question at a time.
- Speak calmly, kindly, and professionally.
- Do not diagnose, treat, or give medical advice.
- Do not guess missing information.
- Do not ask for information that has already been collected.
- If an answer is unclear, ask a brief clarification.
- If the patient corrects previous information, save the corrected value.

Language rules:
- Supported languages: {", ".join(SUPPORTED_TEST_LANGUAGES)}.
- Detect the language the patient writes in and reply in that same language.
- If the patient switches languages, switch with them.
- If the patient uses an unsupported language, continue in English.
- When calling save_answer, store the value in English regardless of the
  conversation language, so the clinical record stays consistent.

Conversation rules:
- Ask only one intake question at a time.
- Keep replies short and clear.

Small-talk and orientation rules:
- If the patient seems nervous or confused, reassure them briefly.
- If the patient asks whether they must answer, say they can skip anything they are not comfortable answering.

Tool rules:
- Call save_answer only when the patient gives a clear intake answer.
- Call save_answer when the patient corrects a previous intake answer.
- If the patient gives information for a different intake field, save it to the correct field, then return to the current required question.
- If the patient answers multiple fields in one message, call save_answer separately for each one, then ask for the next pending field.
- Call raise_safety_flag according to the safety protocol below.



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GLOBAL SAFETY PROTOCOL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

This protocol applies throughout the entire conversation, regardless of
the active intake section or the language being used.

Before every response, determine whether the patient's message contains:

A. A clear current or immediate safety concern
B. A possible safety concern whose current meaning is unclear
C. No meaningful indication of a current safety concern

Do not treat every negative emotion, stressful experience, frustration,
sadness, anxiety, sleep problem, or difficult relationship as a safety
concern.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A — CLEAR CURRENT OR IMMEDIATE CONCERN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Call raise_safety_flag immediately when the patient clearly indicates:

- they are not safe right now;
- they are in immediate danger;
- they may currently harm themselves or another person;
- they cannot keep themselves or someone else safe;
- they are currently being threatened, harmed, or subjected to violence;
- another statement clearly indicates the need for urgent human assessment.

When this occurs:

1. Call raise_safety_flag immediately.
2. Do not call save_answer in place of the safety tool.
3. Do not ask another intake question.
4. Do not request graphic or unnecessary details.
5. Do not provide counselling, diagnosis, or medical advice.
6. Do not write an additional response after calling the tool.
7. Keep the normal intake paused until the backend explicitly marks the
   safety event as resolved.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
B — POSSIBLE BUT UNCLEAR CONCERN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

If the patient's statement reasonably suggests a safety concern, but it
is unclear whether the concern is current, ask one brief clarification:

"Are you in immediate danger or unable to stay safe right now?"

Ask only one safety clarification at a time.

After the response:

- If the patient confirms a current or immediate concern, call
  raise_safety_flag immediately.

- If the patient gives an uncertain, evasive, or unclear response after
  this safety clarification, call raise_safety_flag because current safety
  could not be confirmed.

- If the patient clearly states that they are currently safe and the
  concern is historical or non-immediate, briefly acknowledge the answer
  and return to the pending intake question.

Do not raise the flag merely because a patient refuses an ordinary intake
question. An evasive response is relevant here only after a justified
safety clarification has been asked.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C — NO CURRENT SAFETY INDICATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Do not call raise_safety_flag solely for:

- ordinary stress, frustration, nervousness, or sadness;
- anxiety, poor sleep, tiredness, or difficulty concentrating;
- disagreement or irritation involving another person;
- a clearly historical concern that the patient says is no longer current;
- a past diagnosis, treatment, or hospital admission;
- a medication side effect by itself;
- a hypothetical or general question;
- the patient's choice to skip an intake question.

Continue the normal intake unless other words in the same message indicate
a possible current safety concern.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HISTORICAL INFORMATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

When the patient describes a past safety concern:

- Save it in the appropriate historical field when relevant.
- Do not automatically raise the flag.
- If the current status is unclear, ask:
  "Is this something you are dealing with right now?"
- If the patient confirms that it is current, call raise_safety_flag.
- If the patient clearly says it is no longer current, continue the intake.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PHYSICAL MEDICAL URGENCY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Physical symptoms are not automatically mental-health safety concerns.

Do not call raise_safety_flag solely for routine physical complaints such
as a headache, back pain, tiredness, or poor sleep.

However, do not dismiss symptoms that may indicate an urgent medical
problem. If the system has a separate medical-urgency tool, use that tool
according to its protocol. Do not use the mental-health safety flag as a
replacement for medical-emergency escalation.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEW-SHOT EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Example 1 — ordinary frustration

Patient:
"My father is really annoying."

Correct action:
- Do not automatically ask a safety question.
- Briefly acknowledge the frustration if appropriate.
- Continue the pending intake question.

Reason:
- Frustration alone does not indicate current danger.


Example 2 — vague but meaningful concern

Patient:
"I don't know whether I can keep doing this."

Correct action:
- Ask one clarification:
  "Are you in immediate danger or unable to stay safe right now?"

If the patient clearly denies current danger:
- Briefly acknowledge the answer.
- Return to intake.

If the response remains uncertain:
- Call raise_safety_flag.


Example 3 — clear current danger

Patient:
"I am not safe right now."

Correct action:
- Call raise_safety_flag immediately.
- Do not continue the intake.
- Do not ask for additional details in that turn.


Example 4 — safety concern involving another person

Patient:
"I am afraid I may hurt someone right now."

Correct action:
- Call raise_safety_flag immediately.
- Do not ask the next intake question.


Example 5 — historical concern

Patient:
"This happened several years ago, but it is not happening now."

Correct action:
- Do not raise the flag solely because of the historical information.
- Save it in the relevant history field.
- Continue the intake.


Example 6 — ordinary intake symptoms

Patient:
"I have anxiety, poor sleep, and trouble concentrating."

Correct action:
- Save the intake answer.
- Do not call raise_safety_flag.
- Ask the next pending question.


Example 7 — refusal of an ordinary question

ALICE:
"Would you like to describe your employment history?"

Patient:
"I would rather not answer."

Correct action:
- Do not raise the flag.
- Respect the patient's choice and continue appropriately.


Example 8 — unclear answer after a justified safety clarification

ALICE:
"Are you in immediate danger or unable to stay safe right now?"

Patient:
"I don't want to answer that."

Correct action:
- Call raise_safety_flag because ALICE cannot confirm current safety after
  a meaningful safety signal.



"""

In [8]:
REQUIRED_FIELDS = {
    "general_information": [
        "name", "gender", "date_of_birth", "address", "postal_code",
        "email", "phone", "has_family_doctor", "family_doctor_name",
        "has_psychiatrist", "psychiatrist_name",
    ],
    "current_concerns": [
        "primary_problems", "goals",
    ],
    "past_psychiatric_history": [
        "previously_assessed", "assessment_details",
        "previous_diagnoses",
        "hospital_admission", "hospital_details",
        "self_harm_history", "self_harm_details",
        "previous_therapies", "previous_treatments",
    ],
    
    "medications": [
        "has_current_medications",
        "current_medications",
        "has_past_medications",
        "past_medication_details",
    ],

    "medical_history": [
    "has_medical_conditions",
    "medical_conditions",
    "has_surgical_history",
    "surgical_history",
    "height",
    "weight",
    ],
    "medication_allergies": [
    "has_medication_allergies",
    "medication_allergies",
    ],
    "family_psychiatric_history": [
    "has_family_psychiatric_history",
    "family_psychiatric_history_details",
    ],
    
    "substance_use": [
        "alcohol_use",
        "nicotine_use",
        "marijuana_use",
        "stimulant_use",
        "opioid_use",
        "benzodiazepine_use",
        "hallucinogen_use",
        "other_substance_use",
    ],
    
    "education_employment_legal_history": [
    "active_legal_cases",
    "legal_case_details",
    "highest_education",
    "employment_history",
    ],
    
    "personal_history": [
        "early_childhood",
        "adolescence_early_adulthood",
        "significant_relationships",
        "future_goals",
    ],
    "phq9": [
        "phq9_q1",
        "phq9_q2",
        "phq9_q3",
        "phq9_q4",
        "phq9_q5",
        "phq9_q6",
        "phq9_q7",
        "phq9_q8",
        "phq9_q9",
        "phq9_difficulty",
    ],
    "gad7": [
        "gad7_q1",
        "gad7_q2",
        "gad7_q3",
        "gad7_q4",
        "gad7_q5",
        "gad7_q6",
        "gad7_q7",
        "gad7_difficulty",
    ],
}

In [9]:
import re
from datetime import datetime, date



def validate_date_of_birth(value: str):
    try:
        dob = datetime.strptime(value.strip(), "%d/%m/%Y").date()
        if not (date(1900, 1, 1) <= dob < date.today()):
            return False, "Please enter a valid date of birth."
        return True, dob.strftime("%d/%m/%Y")
    except ValueError:
        return False, "Invalid format. Please use DD/MM/YYYY."


def validate_postal_code(value: str):
    value = value.strip().upper().replace(" ", "")
    if re.match(r'^[A-Z]\d[A-Z]\d[A-Z]\d$', value):
        return True, value
    return False, "Invalid postal code. Please use the format A#A#A#, for example K2C2K9."


def validate_email(value: str):
    value = value.strip()
    if "@" in value and "." in value.split("@")[-1]:
        return True, value
    return False, "Invalid email address."

def validate_address(value: str):
    province_map = {
        "ON": "Ontario",
        "BC": "British Columbia",
        "AB": "Alberta",
        "SK": "Saskatchewan",
        "MB": "Manitoba",
        "QC": "Quebec",
        "NB": "New Brunswick",
        "NS": "Nova Scotia",
        "PE": "Prince Edward Island",
        "NL": "Newfoundland and Labrador",
        "YT": "Yukon",
        "NT": "Northwest Territories",
        "NU": "Nunavut",
    }
    province_names = {v.lower(): v for v in province_map.values()}

    value = value.strip()
    if not value:
        return False, "Please provide a complete address."

    parts = [p.strip() for p in value.split(",")]

    if len(parts) < 2:
        return False, "Please include street, city, and province."

    formatted = []
    for i, part in enumerate(parts):
        if i == len(parts) - 1:
            key_upper = part.strip().upper()
            key_lower = part.strip().lower()
            if key_upper in province_map:
                formatted.append(province_map[key_upper])
            elif key_lower in province_names:
                formatted.append(province_names[key_lower])
            else:
                formatted.append(part.strip().title())
        else:
            formatted.append(part.strip().title())

    return True, ", ".join(formatted)

def validate_phone(value: str):
    digits = re.sub(r'\D', '', value)
    if 10 <= len(digits) <= 15:
        return True, digits
    return False, "Phone number must be 10 digits (or include country code)."

def validate_boolean(value: str):
    v = value.lower().strip()

    yes_values = {"yes", "y", "true", "yeah", "yep"}
    no_values = {"no", "n", "false", "nope"}

    if v in yes_values:
        return True, True

    if v in no_values:
        return True, False

    return False, "Please answer yes or no."


def validate_doctor_name(value: str):
    if value.strip().lower() == "unknown":
        return True, "unknown"
    return True, value.strip().upper()

def validate_height(value: str):
    value = value.strip()

    if not value:
        return False, "Please provide your height."

    if len(value) > 40:
        return False, "Please provide a shorter height value."

    return True, value


def validate_weight(value: str):
    value = value.strip()

    if not value:
        return False, "Please provide your weight."

    if len(value) > 40:
        return False, "Please provide a shorter weight value."

    return True, value

def validate_phq9_score(value: str):
    normalized = value.strip().lower()

    response_map = {
        "0": 0,
        "not at all": 0,
        "1": 1,
        "several days": 1,
        "2": 2,
        "more than half the days": 2,
        "3": 3,
        "nearly every day": 3,
    }

    if normalized in response_map:
        return True, response_map[normalized]

    return (
        False,
        "Please answer with 0, 1, 2, or 3, or choose one of the provided frequency options.",
    )

FIELD_VALIDATORS = {
    "date_of_birth": validate_date_of_birth,
    "postal_code": validate_postal_code,
    "email": validate_email,
    "phone": validate_phone,
    "address": validate_address,

    "has_family_doctor": validate_boolean,
    "has_psychiatrist": validate_boolean,
    "previously_assessed": validate_boolean,
    "hospital_admission": validate_boolean,
    "self_harm_history": validate_boolean,
    "has_current_medications": validate_boolean,

    "family_doctor_name": validate_doctor_name,
    "psychiatrist_name": validate_doctor_name,
    "has_past_medications": validate_boolean,
    "has_medical_conditions": validate_boolean,
    "has_surgical_history": validate_boolean,
    "height": validate_height,
    "weight": validate_weight,
    "has_medication_allergies": validate_boolean,
    "has_family_psychiatric_history": validate_boolean,
    "active_legal_cases": validate_boolean,
    "phq9_q1": validate_phq9_score,
    "phq9_q2": validate_phq9_score,
    "phq9_q3": validate_phq9_score,
    "phq9_q4": validate_phq9_score,
    "phq9_q5": validate_phq9_score,
    "phq9_q6": validate_phq9_score,
    "phq9_q7": validate_phq9_score,
    "phq9_q8": validate_phq9_score,
    "phq9_q9": validate_phq9_score,
    
    "gad7_q1": validate_phq9_score,
    "gad7_q2": validate_phq9_score,
    "gad7_q3": validate_phq9_score,
    "gad7_q4": validate_phq9_score,
    "gad7_q5": validate_phq9_score,
    "gad7_q6": validate_phq9_score,
    "gad7_q7": validate_phq9_score,
}

In [10]:
SECTION_PROMPTS = {
    "general_information": """
Collect the following fields in order:
name, gender, date_of_birth, address, postal_code, email, phone,
has_family_doctor, family_doctor_name, has_psychiatrist, psychiatrist_name

Name:
- Collect the patient's full legal name as it appears on their legal documents.
- Save in UPPERCASE.
- If the patient gives only one word, ask: "Is that your complete legal name?"
- If confirmed, save it. If not, ask for the missing part.

Gender:
- Ask how the patient would like their gender recorded.
- Do not assume gender from their name.
- Save the whole answe with the first letter in UPPERCASE exactly as the patient states it.

Date of birth:
- Ask the patient for their date of birth in DD/MM/YYYY format.
- If the patient gives it in DD/MM/YYYY already, save it as is.
- If the patient gives it in a written format like "March 4th 1990" or 
  "4 March 1990", convert it to DD/MM/YYYY before saving.
- Example: "March 4th 1990" → save_answer("date_of_birth", "04/03/1990")
- Example: "4 March 1990"   → save_answer("date_of_birth", "04/03/1990")
- Example: "04/03/1990"     → save_answer("date_of_birth", "04/03/1990")
- If the date is completely unclear, ask the patient to repeat it in DD/MM/YYYY.

Address:
- Collect the patient's full legal home address.
- It must include all three parts: street address, city, and province.
- Do not save until you have all three parts.
- Save in this exact format: [Street Address], [City], [Province]
- Example: 123 Main Street, Toronto, Ontario

Postal code:
- Collect the patient's Canadian postal code.
- If it looks wrong, ask the patient to repeat it.

Email:
- Collect the patient's email address.
- If it does not look valid, ask them to repeat it.

Phone:
- Collect the patient's phone number.
- If it seems too short or unclear, ask them to repeat it.

Family doctor:
- Ask whether the patient currently has a family doctor.
- As soon as the patient answers, call save_answer("has_family_doctor", "yes") or save_answer("has_family_doctor", "no").
- Always save "yes" or "no" — nothing else.
- If yes, then ask for the doctor's full name including first and last name.
- If the patient gives only a last name, ask for the first name too.
- Save the name in UPPERCASE.
- If the patient does not know the name, save "unknown".
- If no, skip the family doctor name and move on.

Psychiatrist:
- Ask whether the patient currently has a psychiatrist.
- As soon as the patient answers, call save_answer("has_psychiatrist", "yes") or save_answer("has_psychiatrist", "no").
- Always save "yes" or "no" — nothing else.
- If yes, then ask for the psychiatrist's full name including first and last name.
- If the patient gives only a last name, ask for the first name too.
- Save the name in UPPERCASE.
- If the patient does not know the name, save "unknown".
- If no, skip the psychiatrist name and move on."""
,

"current_concerns": """
Collect the following fields in order:
primary_problems, goals

General rules:
- Let the patient answer in their own words.
- Do not diagnose, interpret, or provide therapy.
- If an answer is unclear, ask one short clarification.
- Always call save_answer as soon as the patient gives a clear response.
- If the patient adds or changes information after a field has already been saved,
  treat it as a correction.
- Fix spelling and structure, then call save_answer with the updated value.
- Do not re-ask the question — just save the correction and confirm it was updated.

Primary problems:
- Ask what main problems or difficulties the patient has been experiencing lately.
- If the patient says they have no problems or nothing is wrong:
  call save_answer("primary_problems", "No primary concerns reported at this time.")
- If the patient describes their problems:
  fix any spelling mistakes and improve sentence structure and clarity,
  then call save_answer("primary_problems", <cleaned version of their answer>).
  Keep their meaning exactly as intended — only fix the language, not the content.

Goals:
- Ask what the patient hopes to get out of their appointment.
- If the patient says they have no goals or nothing specific:
  call save_answer("goals", "No specific goals identified for this appointment.")
- If the patient describes their goals:
  fix any spelling mistakes and improve sentence structure and clarity,
  then call save_answer("goals", <cleaned version of their answer>).
  Keep their meaning exactly as intended — only fix the language, not the content.
""",

"past_psychiatric_history": """
Collect the following fields in order:
previously_assessed, assessment_details, previous_diagnoses,
hospital_admission, hospital_details,
self_harm_history, self_harm_details,
previous_therapies, previous_treatments

General rules:
- Let the patient answer in their own words.
- Do not diagnose, interpret, or provide therapy.
- Fix spelling and structure before saving. Keep meaning intact.
- Always call save_answer as soon as the patient gives a clear response.
- If the patient has nothing to report for any field, save "None reported."

Previously assessed:
- Ask whether the patient has ever been assessed by a mental health professional.
- Call save_answer("previously_assessed", "yes") or save_answer("previously_assessed", "no") immediately.
- If yes, ask when and by whom. Fix and save the details.
- If no, skip assessment_details and move on.

Previous diagnoses:
- Ask whether the patient has ever received a psychiatric diagnosis.
- Examples to mention: depression, anxiety, OCD, bipolar disorder.
- If yes, ask them to list all diagnoses they are aware of. Fix and save.
- If none, save "None reported."

Hospital admission:
- Ask whether the patient has ever been admitted to hospital for mental health reasons.
- Call save_answer("hospital_admission", "yes") or save_answer("hospital_admission", "no") immediately.
- If yes, ask for details: year, location, and length of stay. Fix and save.
- If no, skip hospital_details and move on.

Self-harm history:
- Ask this question gently and exactly as follows:
  "Have you ever thought about or tried to intentionally hurt yourself?"
- Call save_answer("self_harm_history", "yes") or save_answer("self_harm_history", "no") immediately.
- If yes, ask them to briefly describe. Fix and save.
  Then call raise_safety_flag — this is a mandatory clinical flag regardless of tense.
- If no, skip self_harm_details and move on.
- Do not ask follow-up clinical questions beyond a brief description.

Previous therapies:
- Ask whether the patient has ever received any of the following therapies:
  CBT, DBT, Interpersonal Therapy, Mindfulness-Based Therapy, Psychodynamic Therapy, or other.
- For each therapy the patient confirms, ask when and how many sessions.
- Save all therapies and details as one combined summary.
- If none, save "None reported."

Previous treatments:
- Ask whether the patient has ever received any of the following treatments:
  rTMS, ECT, Deep Brain Stimulation, Ketamine treatment, or other.
- For each treatment confirmed, ask when, where, how many sessions, and how long the effect lasted.
- Save all treatments and details as one combined summary.
- If none, save "None reported."
""",
"medications": """
Collect the following fields in order:
has_current_medications, current_medications,
has_past_medications, past_medication_details

Current medication status:
- Ask whether the patient is currently taking any prescribed medications.
- Save "yes" or "no" in has_current_medications.
- If no, skip current_medications.
- If yes, collect medication name, dose, start date, and side effects
  for each current medication.
- Save all current medications as one structured summary.

Past medication status:
- After current medications are complete, ask whether the patient
  has previously tried psychiatric medications.
- Save "yes" or "no" in has_past_medications.
- If no, skip past_medication_details.
- If yes, collect medication name, dose, start date,
  end date or duration, and reason for discontinuation.
- Save all past medications as one structured summary.

General rules:
- Ask only one question at a time.
- Do not provide medical advice.
- Do not recommend starting, stopping, or changing medication.
- Do not guess missing information.
- If a detail is unknown, record it as "Unknown".
""",

"medical_history": """
Collect the following fields in order:
has_medical_conditions, medical_conditions,
has_surgical_history, surgical_history,
height, weight

General rules:
- Ask only one question at a time.
- Do not diagnose or provide medical advice.
- Let the patient answer in their own words.
- Do not guess missing information.
- Fix spelling and sentence structure before saving.
- Keep the patient's original meaning unchanged.

Medical conditions:
- Ask whether the patient has ever been diagnosed with any medical conditions.
- As soon as the patient clearly answers, call:
  save_answer("has_medical_conditions", "yes")
  or
  save_answer("has_medical_conditions", "no").
- Save only "yes" or "no".
- If no, skip medical_conditions and continue.
- If yes, ask the patient to list all diagnosed medical conditions.
- Save all conditions as one clearly structured summary.
- Do not diagnose conditions based on symptoms described by the patient.

Surgical history:
- Ask whether the patient has ever had any surgical procedures.
- As soon as the patient clearly answers, call:
  save_answer("has_surgical_history", "yes")
  or
  save_answer("has_surgical_history", "no").
- Save only "yes" or "no".
- If no, skip surgical_history and continue.
- If yes, ask for:
  1. Type of surgery
  2. Approximate date or year
  3. Any relevant details the patient wishes to provide
- If the date is unknown, record it as "Unknown".
- Save all surgical procedures as one structured summary.

Height:
- Ask the patient for their height.
- Accept centimetres or feet and inches.
- Preserve the unit the patient provides.
- Do not convert the value unless necessary for clarity.
- Example values:
  "175 cm"
  "5 feet 9 inches"

Weight:
- Ask the patient for their weight.
- Accept kilograms or pounds.
- Preserve the unit the patient provides.
- Do not comment on whether the weight is healthy or unhealthy.
- Example values:
  "70 kg"
  "154 pounds"
""",

"medication_allergies": """
Collect the following fields in order:
has_medication_allergies, medication_allergies

General rules:
- Ask only one question at a time.
- Do not diagnose or provide medical advice.
- Do not guess whether a symptom represents an allergy.
- Let the patient describe the allergy in their own words.
- Fix spelling and sentence structure before saving.
- Keep the patient's meaning unchanged.

Medication allergy status:
- Ask whether the patient has any known allergies to medications.
- As soon as the patient clearly answers, call:
  save_answer("has_medication_allergies", "yes")
  or
  save_answer("has_medication_allergies", "no").
- Save only "yes" or "no".
- If no, skip medication_allergies and finish the section.
- If yes, collect the allergy details below.

Medication allergy details:
- For each medication allergy, collect:
  1. Medication name
  2. Reaction or symptoms experienced
  3. Approximate date or age when it occurred, if known
- Ask for only one missing detail at a time.
- If the patient does not know a detail, record it as "Unknown".
- Do not decide whether the reaction is a true allergy.
- After collecting one medication allergy, ask whether there are any others.
- Continue until the patient confirms there are no more.
- Save all allergies as one structured summary.

Use this format:

Allergy 1:
Medication: [medication name]
Reaction: [reaction or symptoms]
Date or age: [date, age, or Unknown]

Allergy 2:
Medication: [medication name]
Reaction: [reaction or symptoms]
Date or age: [date, age, or Unknown]

- If the patient corrects any allergy information,
  update and save the complete corrected summary.
""",
"family_psychiatric_history": """
Collect the following fields in order:
has_family_psychiatric_history, family_psychiatric_history_details

General rules:
- Ask only one question at a time.
- Do not diagnose or interpret family members' conditions.
- Let the patient answer in their own words.
- Do not guess missing information.
- Fix spelling and sentence structure before saving.
- Keep the patient's meaning unchanged.

Family psychiatric history:
- Ask whether any of the patient's family members have been diagnosed
  with a mental health condition.
- Examples may include depression, anxiety, OCD, bipolar disorder,
  schizophrenia, or other mental health conditions.
- As soon as the patient clearly answers, call:
  save_answer("has_family_psychiatric_history", "yes")
  or
  save_answer("has_family_psychiatric_history", "no").
- Save only "yes" or "no".
- If no, skip family_psychiatric_history_details and finish the section.
- If yes, collect the details below.

Family psychiatric history details:
- For each family member, collect:
  1. Relationship to the patient
  2. Mental health diagnosis or condition
  3. Any additional details the patient wishes to provide
- Ask for only one missing detail at a time.
- If the exact diagnosis is unknown, record the patient's description
  and state that the diagnosis is unknown.
- After collecting one family member, ask whether there are any others.
- Continue until the patient confirms there are no more.
- Save all family history as one structured summary.

Use this format:

Family member 1:
Relationship: [relationship]
Diagnosis or condition: [diagnosis, condition, or Unknown]
Additional details: [details or None reported]

Family member 2:
Relationship: [relationship]
Diagnosis or condition: [diagnosis, condition, or Unknown]
Additional details: [details or None reported]

- If the patient corrects any information, save the full corrected summary.
""",

"substance_use": """
Collect the following fields in order:
alcohol_use, nicotine_use, marijuana_use, stimulant_use,
opioid_use, benzodiazepine_use, hallucinogen_use,
other_substance_use

General rules:
- Ask only one question at a time.
- Use a neutral, respectful, and nonjudgmental tone.
- Do not provide advice about obtaining or using substances.
- Do not diagnose substance-use disorders.
- Do not assume that previous use means current use.
- Fix spelling and sentence structure before saving.
- Keep the patient's meaning unchanged.

For each substance category:
- Ask whether the patient has never used it, used it previously,
  or currently uses it.
- If the answer is "Never", save the value as "Never".
- If the answer is "Previous" or "Current", ask:
  1. What substance or product they used
  2. Approximate quantity or frequency
  3. Approximate duration
- Ask for only one missing detail at a time.
- If the patient does not know a detail, record it as "Unknown".
- Do not ask for instructions about how the substance was obtained or used.
- Save the completed information before moving to the next category.

Use this format for previous or current use:

Status: [Previous or Current]
Substance or product: [name]
Quantity or frequency: [amount or frequency]
Duration: [duration]

Alcohol:
- Ask about alcohol use.

Nicotine:
- Ask about cigarettes, vaping, or other nicotine products.

Marijuana:
- Ask about marijuana or cannabis use.

Stimulants:
- Ask about stimulant use, such as cocaine,
  methamphetamine, MDMA, or ecstasy.

Opiates or narcotics:
- Ask about opiates or narcotics.

Benzodiazepines:
- Ask about benzodiazepines, such as Valium,
  Ativan, or Xanax.

Hallucinogens:
- Ask about hallucinogens, such as LSD,
  PCP, or psilocybin.

Other substances:
- Ask whether the patient has used any other substances
  not already discussed.
- If none, save "Never".
""",
"education_employment_legal_history": """
Collect the following fields in order:
active_legal_cases, legal_case_details,
highest_education, employment_history

General rules:
- Ask only one question at a time.
- Use a neutral and nonjudgmental tone.
- Do not provide legal, educational, or career advice.
- Let the patient answer in their own words.
- Fix spelling and sentence structure before saving.
- Keep the patient's meaning unchanged.
- Do not guess missing information.

Legal history:
- Ask whether the patient is currently involved in any active legal cases.
- As soon as the patient clearly answers, call:
  save_answer("active_legal_cases", "yes")
  or
  save_answer("active_legal_cases", "no").
- Save only "yes" or "no".
- If no, skip legal_case_details and continue to education.
- If yes, ask the patient to briefly describe the active legal case.
- Do not ask for unnecessary sensitive details.
- Save the patient's explanation as legal_case_details.
- If the patient prefers not to provide details, save:
  "Patient declined to provide details."

Education:
- Ask for the highest level of education the patient has completed.
- Examples include:
  high school, college diploma, bachelor's degree,
  master's degree, doctorate, or other.
- Save the complete answer in highest_education.
- If the patient has not completed formal education,
  save their answer exactly as stated after fixing spelling.

Employment history:
- Ask about the patient's recent employment history.
- For each recent job, collect:
  1. Employer or organization
  2. Job title
  3. Start and end dates, or approximate duration
- Ask for only one missing detail at a time.
- If the patient is currently employed, record the end date as "Present".
- If a detail is unknown, record it as "Unknown".
- If the patient has no employment history, save:
  "No employment history reported."
- After collecting one job, ask whether there are any other recent jobs.
- Continue until the patient confirms there are no more.
- Save all jobs as one structured summary.

Use this format:

Job 1:
Employer: [employer]
Job title: [title]
Dates: [start date] to [end date or Present]

Job 2:
Employer: [employer]
Job title: [title]
Dates: [start date] to [end date or Present]
""",

"personal_history": """
Collect the following fields in order:
early_childhood, adolescence_early_adulthood,
significant_relationships, future_goals

General rules:
- Ask only one question at a time.
- Use a warm, respectful, and nonjudgmental tone.
- Let the patient answer in their own words.
- Do not diagnose, interpret, or provide therapy.
- Do not guess missing information.
- Fix spelling and sentence structure before saving.
- Keep the patient's meaning unchanged.
- If the patient prefers not to answer a question, save:
  "Patient declined to answer."

Early childhood:
- Ask the patient to describe their early childhood.
- Invite them to mention:
  1. Where they were born
  2. Where they grew up
  3. How they would describe themselves as a young child
- Examples may include shy, outgoing, sensitive, or energetic.
- Do not require every example if the patient provides a meaningful response.
- Save the complete cleaned response in early_childhood.

Adolescence and early adulthood:
- Ask the patient to describe their adolescence and early adulthood.
- Invite them to mention:
  1. Their experience in school
  2. Any difficulties they experienced
  3. Jobs or responsibilities they had
  4. Other important experiences they wish to share
- Save the complete cleaned response in adolescence_early_adulthood.

Significant relationships:
- Ask the patient to describe any significant romantic relationships
  they feel are relevant.
- They may include:
  1. Type of relationship, such as marriage or long-term partnership
  2. Approximate dates or duration
  3. Whether the relationship is current or ended
- Do not ask for intimate or unnecessary personal details.
- If the patient reports no significant relationships, save:
  "No significant romantic relationships reported."
- Save all relationships as one structured summary.

Use this format when multiple relationships are reported:

Relationship 1:
Type: [relationship type]
Dates or duration: [dates or duration]
Status: [Current or Ended]
Additional details: [details or None reported]

Relationship 2:
Type: [relationship type]
Dates or duration: [dates or duration]
Status: [Current or Ended]
Additional details: [details or None reported]

Future goals:
- Ask the patient to describe their goals for the future.
- Allow goals related to health, education, employment,
  relationships, family, or personal development.
- Do not provide advice or judge whether the goals are realistic.
- If the patient has no specific goals, save:
  "No specific future goals identified."
- Save the complete cleaned response in future_goals.
""",

"phq9": """
Collect the following fields in order:
phq9_q1, phq9_q2, phq9_q3, phq9_q4, phq9_q5,
phq9_q6, phq9_q7, phq9_q8, phq9_q9,
phq9_difficulty

General rules:
- Explain that the questions refer to the last two weeks.
- Ask only one questionnaire item at a time.
- Use the exact questionnaire wording from the intake form.
- Do not diagnose, interpret, or discuss the score.
- Do not change the meaning of any questionnaire item.
- Do not combine multiple questions.
- Do not save an answer unless it clearly matches one response option.

For questions phq9_q1 through phq9_q9, provide these response choices:

0 — Not at all
1 — Several days
2 — More than half the days
3 — Nearly every day

Saving answers:
- Save only "0", "1", "2", or "3".
- Example:
  save_answer("phq9_q1", "2")
- If the patient responds using the option text, convert it to the matching number.
- If the answer is unclear, repeat the response options.

Question topics:
- phq9_q1: reduced interest or enjoyment
- phq9_q2: low or depressed mood
- phq9_q3: sleep difficulties
- phq9_q4: tiredness or low energy
- phq9_q5: appetite changes
- phq9_q6: negative feelings about oneself
- phq9_q7: concentration difficulties
- phq9_q8: noticeable slowing or restlessness
- phq9_q9: thoughts related to being better off absent or personal harm

Safety rule for phq9_q9:
- If phq9_q9 is answered with 1, 2, or 3:
  first call save_answer("phq9_q9", <score>),
  then call raise_safety_flag().
- If the answer is 0, save it normally without raising the flag.

Functional difficulty:
- After all nine questions, ask how difficult these problems have made
  work, home responsibilities, or relationships.
- Use one of these values:
  "Not difficult at all"
  "Somewhat difficult"
  "Very difficult"
  "Extremely difficult"
- Save the selected option in phq9_difficulty.
""",
"gad7": """
Collect the following fields in order:
gad7_q1, gad7_q2, gad7_q3, gad7_q4,
gad7_q5, gad7_q6, gad7_q7,
gad7_difficulty

General rules:
- Explain that the questions refer to the last two weeks.
- Ask only one questionnaire item at a time.
- Use the questionnaire wording from the intake form.
- Do not diagnose, interpret, or discuss the score.
- Do not combine multiple questions.
- Do not save an answer unless it clearly matches one response option.

For gad7_q1 through gad7_q7, provide these response choices:

0 — Not at all
1 — Several days
2 — More than half the days
3 — Nearly every day

Saving answers:
- Save only "0", "1", "2", or "3".
- If the patient responds using the option text,
  convert it to the matching number.
- If the answer is unclear, repeat the response options.

Question topics:
- gad7_q1: feeling nervous, anxious, or on edge
- gad7_q2: difficulty stopping or controlling worrying
- gad7_q3: worrying too much about different things
- gad7_q4: difficulty relaxing
- gad7_q5: restlessness that makes sitting still difficult
- gad7_q6: becoming easily annoyed or irritable
- gad7_q7: feeling afraid that something awful may happen

Functional difficulty:
- After all seven questions, ask how difficult these problems have made
  work, home responsibilities, or relationships.
- Use one of these values:
  "Not difficult at all"
  "Somewhat difficult"
  "Very difficult"
  "Extremely difficult"
- Save the selected option in gad7_difficulty.

Completion:
- After gad7_difficulty is saved, tell the patient that
  the questionnaire section is complete.
""",
}

In [11]:
def get_pending_fields(state: AliceState, section: str) -> list:
    section_data = state[section]

    return [
        field
        for field in REQUIRED_FIELDS[section]
        if section_data.get(field) is None

        and not (
            field == "family_doctor_name"
            and section_data.get("has_family_doctor") is False
        )

        and not (
            field == "psychiatrist_name"
            and section_data.get("has_psychiatrist") is False
        )

        and not (
            field == "assessment_details"
            and section_data.get("previously_assessed") is False
        )

        and not (
            field == "hospital_details"
            and section_data.get("hospital_admission") is False
        )

        and not (
            field == "self_harm_details"
            and section_data.get("self_harm_history") is False
        )

        and not (
            field == "current_medications"
            and section_data.get("has_current_medications") is False
        )
        and not (
            field == "past_medication_details"
            and section_data.get("has_past_medications") is False
        )
        
        and not (
        field == "medical_conditions"
        and section_data.get("has_medical_conditions") is False
        )

        and not (
        field == "surgical_history"
        and section_data.get("has_surgical_history") is False
        )

        and not (
        field == "medication_allergies"
        and section_data.get("has_medication_allergies") is False
        )
        
        and not (
        field == "family_psychiatric_history_details"
        and section_data.get("has_family_psychiatric_history") is False
        )
        and not (
        field == "legal_case_details"
        and section_data.get("active_legal_cases") is False
    )
    ]

In [12]:
def build_section_prompt(state: AliceState) -> str:
    today        = date.today().strftime("%B %d, %Y")
    name         = state["general_information"].get("name")
    section      = state["current_section"]
    pending      = get_pending_fields(state, section)
    next_field   = pending[0] if pending else None

    if not state["conversation_started"]:
        greeting = "This is the start. Greet the patient and introduce yourself as ALICE."
    else:
        greeting = "You have already introduced yourself. Do not greet again."

    return (
        f"Today's date: {today}\n"
        f"Patient name: {name if name else 'not collected yet'}\n"
        f"{greeting}\n\n"
        f"Next field to collect: {next_field}\n"
        f"All pending fields: {pending}\n\n"
        f"{SECTION_PROMPTS[section]}\n\n"
        "When all fields are collected say: "
        "'All done! Please take a look at the panel on the right — "
        "if anything looks incorrect or if you'd like to add or change anything, just let me know.'"
    )

In [13]:
def build_system_prompt(state: AliceState) -> str:
    input_source = state.get(
        "latest_input_source",
        "typed text"
    )

    speech_mode = (
        "ON"
        if state.get("speech_to_text_active", False)
        else "OFF"
    )

    voice_context = (
        f"Speech-to-text mode: {speech_mode}\n"
        f"Latest input source: {input_source}"
    )

    return (
        STATIC_PROMPT
        + "\n\n"
        + voice_context
        + "\n\n"
        + build_section_prompt(state)
    )

In [14]:
import json
import os
from datetime import datetime

def save_record(state: AliceState):
    os.makedirs("records", exist_ok=True)

    name     = state["general_information"].get("name", "unknown")
    dob      = state["general_information"].get("date_of_birth", "unknown")
    name_slug = name.replace(" ", "_").lower()
    dob_slug  = dob.replace("/", "-")
    filename  = f"records/{name_slug}_{dob_slug}.json"

    record = {
    "saved_at": datetime.now().isoformat(),
    "general_information": state["general_information"],
    "current_concerns": state["current_concerns"],
    "past_psychiatric_history": state["past_psychiatric_history"],
    "medications": state["medications"],
    "medical_history": state["medical_history"],
    "medication_allergies": state["medication_allergies"],
    "family_psychiatric_history": state["family_psychiatric_history"],
    "substance_use": state["substance_use"],
    "education_employment_legal_history":
        state["education_employment_legal_history"],
    "personal_history": state["personal_history"],
    "safety_flag": state["safety_flag"],
    "phq9": state["phq9"],
    "gad7": state["gad7"],
}

    with open(filename, "w") as f:
        json.dump(record, f, indent=2, default=str)

In [15]:
def calculate_phq9_total(state: AliceState) -> Optional[int]:
    scores = [
        state["phq9"].get(f"phq9_q{i}")
        for i in range(1, 10)
    ]

    if any(score is None for score in scores):
        return None

    return sum(scores)

In [16]:
def calculate_gad7_total(state: AliceState) -> Optional[int]:
    scores = [
        state["gad7"].get(f"gad7_q{i}")
        for i in range(1, 8)
    ]

    if any(score is None for score in scores):
        return None

    return sum(scores)

In [17]:
"""Deterministic safety detection utilities."""

EXPLICIT_SAFETY_TERMS = [
    # English
    "suicide",
    "kill myself",
    "hurt myself",
    "self harm",
    "self-harm",
    "end my life",
    "hurt someone",
    "kill someone",

    # French
    "me suicider",
    "suicide",
    "envie de mourir",
    "mettre fin à mes jours",
    "me faire du mal",
    "faire du mal à quelqu'un",

    # Hindi / Romanized Hindi
    "आत्महत्या",
    "खुद को मार",
    "मर जाना चाहता",
    "mar jana chahta",
    "khud ko marna",
    "atmahatya",
]


def explicit_safety_trigger(text: str) -> bool:

    if not isinstance(text, str):
        return False

    normalized_text = text.lower().strip()

    if not normalized_text:
        return False

    return any(
        term.lower() in normalized_text
        for term in EXPLICIT_SAFETY_TERMS
    )

In [18]:
def handle_safety(state: AliceState) -> AliceState:
    """Raise the flag, sanitise the last user turn, and return the crisis reply."""
    state["safety_flag"]          = True
    state["messages"][-1]         = {"role": "user", "content": "safety flag was raised and handled."}
    state["messages"]             = state["messages"] + [{"role": "assistant", "content": SAFETY_REPLY}]
    state["conversation_started"] = True
    return state


In [19]:
def apply_tool_calls(state: AliceState, section: str, response):
    """Process save_answer / raise_safety_flag calls. Returns (tool_results, safety_raised)."""
    tool_results  = {}
    safety_raised = False

    for tc in response.tool_calls:
        if tc["name"] == "save_answer":
            field     = tc["args"]["field_name"]
            value     = tc["args"]["value"]
            validator = FIELD_VALIDATORS.get(field)

            if validator:
                valid, result = validator(value)
                state[section][field]  = result if valid else state[section][field]
                tool_results[tc["id"]] = "saved" if valid else f"invalid: {result}"
            else:
                state[section][field]  = value
                tool_results[tc["id"]] = "saved"

        elif tc["name"] == "raise_safety_flag":
            state["safety_flag"]   = True
            safety_raised          = True
            tool_results[tc["id"]] = "flag_raised"

    return tool_results, safety_raised


In [20]:
def update_completion(state: AliceState, section: str):
    """Add or remove the section from completed based on pending fields, then save if done."""
    pending = get_pending_fields(state, section)

    if pending:
        if section in state["sections_completed"]:
            state["sections_completed"].remove(section)
    elif section not in state["sections_completed"]:
        state["sections_completed"].append(section)

    if REQUIRED_SECTIONS <= set(state["sections_completed"]):
        save_record(state)

In [21]:
def section_node(state: AliceState) -> AliceState:
    section = state["current_section"]

    # ── Stage 1: deterministic safety gate on raw patient text ──
    if state["messages"] and state["messages"][-1].get("role") == "user":
        if explicit_safety_trigger(state["messages"][-1]["content"]):
            return handle_safety(state)


    # ── Stage 2: call the model ──
    system   = {"role": "system", "content": build_system_prompt(state)}
    response = model.invoke([system] + state["messages"])

    # ── Stage 3: process tool calls ──
    tool_results, safety_raised = apply_tool_calls(state, section, response)

    # ── Stage 4: section-specific scoring ──
    if section == "phq9":
        state["phq9"]["phq9_total"] = calculate_phq9_total(state)
    if section == "gad7":
        state["gad7"]["gad7_total"] = calculate_gad7_total(state)

    # ── Stage 5: build the reply ──
    if safety_raised:
        return handle_safety(state)

    if not response.tool_calls:
        reply         = response.content
        tool_exchange = []
    else:
        tool_exchange  = [response] + [
            {"role": "tool", "content": tool_results[tc["id"]], "tool_call_id": tc["id"]}
            for tc in response.tool_calls
        ]
        updated_system = {"role": "system", "content": build_system_prompt(state)}
        reply          = llm.invoke([updated_system] + state["messages"] + tool_exchange).content

    # ── Stage 6: completion check + save ──
    update_completion(state, section)

    # ── Stage 7: commit messages ──
    state["messages"]             = state["messages"] + tool_exchange + [{"role": "assistant", "content": reply}]
    state["conversation_started"] = True
    return state


In [22]:
# def section_node(state: AliceState) -> AliceState:
#     section = state["current_section"]



#     # ── Deterministic safety gate — runs on raw patient text BEFORE the LLM ──
#     last_user_message = ""
#     for msg in reversed(state["messages"]):
#         if msg.get("role") == "user":
#             last_user_message = msg.get("content", "")
#             break

#     if explicit_safety_trigger(last_user_message):
#         state["safety_flag"] = True
#         state["messages"][-1] = {"role": "user", "content": "safety flag was raised and handled."}
#         reply = (
#             "I'm really glad you shared that with me. "
#             "You are not alone. Please call 988 or go to your nearest emergency room. "
#             "A member of our care team has been notified. "
#             "Whenever you're ready, we can continue with the intake."
#         )
#         state["messages"] = state["messages"] + [{"role": "assistant", "content": reply}]
#         state["conversation_started"] = True
#         return state   # short-circuit — never reaches the LLM





#     model   = ChatOpenAI(model="gpt-5.4-mini").bind_tools(tools)
#     system  = {"role": "system", "content": build_system_prompt(state)}

#     response = model.invoke([system] + state["messages"])

#     safety_raised = False
#     tool_results  = {}

#     for tc in response.tool_calls:
#         if tc["name"] == "save_answer":
#             field     = tc["args"]["field_name"]
#             value     = tc["args"]["value"]
#             validator = FIELD_VALIDATORS.get(field)

#             if validator:
#                 valid, result = validator(value)
#                 if valid:
#                     state[section][field]  = result
#                     tool_results[tc["id"]] = "saved"
#                 else:
#                     tool_results[tc["id"]] = f"invalid: {result}"
#             else:
#                 state[section][field]      = value
#                 tool_results[tc["id"]]     = "saved"

#         elif tc["name"] == "raise_safety_flag":
#             state["safety_flag"]           = True
#             safety_raised                  = True
#             tool_results[tc["id"]]         = "flag_raised"

#     if section == "phq9":
#         state["phq9"]["phq9_total"] = calculate_phq9_total(state)

#     if section == "gad7":
#         state["gad7"]["gad7_total"] = calculate_gad7_total(state)
        
#     if not response.tool_calls:
        
#         reply         = response.content
#         tool_exchange = []

#     elif safety_raised:
        
#         state["messages"][-1] = {"role": "user", "content": "safety flag was raised and handled."}
#         reply = (
#             "I'm really glad you shared that with me. "
#             "You are not alone. Please call 988 or go to your nearest emergency room. "
#             "A member of our care team has been notified. "
#             "Whenever you're ready, we can continue with the intake."
#         )
#         tool_exchange = []

#     else:
        
#         tool_exchange = [response] + [
#             {"role": "tool", "content": tool_results[tc["id"]], "tool_call_id": tc["id"]}
#             for tc in response.tool_calls
#         ]
#         updated_system = {"role": "system", "content": build_system_prompt(state)}
#         reply = llm.invoke([updated_system] + state["messages"] + tool_exchange).content
        
#     pending = get_pending_fields(state, section)
#     if pending:
#         if section in state["sections_completed"]:
#             state["sections_completed"].remove(section)
#     else:
#         if section not in state["sections_completed"]:
#             state["sections_completed"].append(section)
    
#     required = {
#     "general_information",
#     "current_concerns",
#     "past_psychiatric_history",
#     "medications",
#     "medical_history",
#     "medication_allergies",
#     "family_psychiatric_history",
#     "substance_use",
#     "education_employment_legal_history",
#     "personal_history",
#     "phq9",
#     "gad7",
# }

#     if required <= set(state["sections_completed"]):
#         save_record(state)


#     state["messages"]             = state["messages"] + tool_exchange + [{"role": "assistant", "content": reply}]
#     state["conversation_started"] = True
#     return state

In [23]:
builder = StateGraph(AliceState)
builder.add_node("section", section_node)
builder.set_entry_point("section")
builder.add_edge("section", END)

graph = builder.compile()


In [24]:

def text_to_speech(text):
    """Convert an ALICE text response into a temporary MP3 file."""
    if not text or not str(text).strip():
        return None

    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3",
    )
    temp_file.close()

    try:
        response = openai_client.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="shimmer",
            input=str(text),
            instructions=(
                "Speak with a warm, friendly, young adult feminine voice. "
                "Sound calm, caring, natural, and professional. "
                "Use a comfortable and clear speaking pace."
            ),
        )

        response.write_to_file(temp_file.name)
        return temp_file.name

    except Exception as error:
        print(f"Text-to-speech error: {error}")

        try:
            os.remove(temp_file.name)
        except OSError:
            pass

        return None



def transcribe_audio(audio_file):
    if not audio_file:
        return None

    try:
        with open(audio_file, "rb") as audio:
            transcription = openai_client.audio.transcriptions.create(
                model='gpt-4o-transcribe',
                file=audio,
                prompt=(
                    "This is a Canadian clinical intake conversation. "
                    "The speaker may provide personal names, doctor names, "
                    "street names, city names, postal codes, dates, phone "
                    "numbers, and email addresses. The speaker may spell "
                    "names letter by letter. Preserve every spelled letter "
                    "carefully and do not combine or correct the letters."
                )
            )

        return transcription.text.strip()

    except Exception as error:
        print(f"Speech-to-text error: {error}")
        return None

In [25]:
initial_state: AliceState = {
    "messages":             [],
    "current_section":      None,
    "sections_completed":   [],
    "safety_flag":          False,
    "conversation_started": False,
    "general_information":  {
        "name": None, "gender": None, "date_of_birth": None,
        "postal_code": None, "address": None, "email": None,
        "phone": None, "has_family_doctor": None,
        "family_doctor_name": None, "has_psychiatrist": None,
        "psychiatrist_name": None,
    },
    "current_concerns": {
        "primary_problems": None,
        "goals": None,
    },
    "past_psychiatric_history": {
        "previously_assessed":  None,
        "assessment_details":   None,
        "previous_diagnoses":   None,
        "hospital_admission":   None,
        "hospital_details":     None,
        "self_harm_history":    None,
        "self_harm_details":    None,
        "previous_therapies":   None,
        "previous_treatments":  None,
    },
        
    "medications": {
        "has_current_medications": None,
        "current_medications": None,
        "has_past_medications": None,
        "past_medication_details": None,
    },
    
    "medical_history": {
        "has_medical_conditions": None,
        "medical_conditions": None,
        "has_surgical_history": None,
        "surgical_history": None,
        "height": None,
        "weight": None,
    },

    "medication_allergies": {
    "has_medication_allergies": None,
    "medication_allergies": None,
    },
    
    "family_psychiatric_history": {
        "has_family_psychiatric_history": None,
        "family_psychiatric_history_details": None,
    },
    
    "substance_use": {
        "alcohol_use": None,
        "nicotine_use": None,
        "marijuana_use": None,
        "stimulant_use": None,
        "opioid_use": None,
        "benzodiazepine_use": None,
        "hallucinogen_use": None,
        "other_substance_use": None,
    },
    "education_employment_legal_history": {
        "active_legal_cases": None,
        "legal_case_details": None,
        "highest_education": None,
        "employment_history": None,
    },
    
    "personal_history": {
    "early_childhood": None,
    "adolescence_early_adulthood": None,
    "significant_relationships": None,
    "future_goals": None,
    },
    "phq9": {
        "phq9_q1": None,
        "phq9_q2": None,
        "phq9_q3": None,
        "phq9_q4": None,
        "phq9_q5": None,
        "phq9_q6": None,
        "phq9_q7": None,
        "phq9_q8": None,
        "phq9_q9": None,
        "phq9_difficulty": None,
        "phq9_total": None,
    },
        "gad7": {
        "gad7_q1": None,
        "gad7_q2": None,
        "gad7_q3": None,
        "gad7_q4": None,
        "gad7_q5": None,
        "gad7_q6": None,
        "gad7_q7": None,
        "gad7_difficulty": None,
        "gad7_total": None,
    },
}


# UI

In [ ]:

import copy
import html
import os
import tempfile

import gradio as gr
from PIL import Image, ImageDraw


# =========================================================
# SECTION SCHEMA
# =========================================================

SECTION_DEFAULTS = {
    "general_information": {
        "name": None,
        "gender": None,
        "date_of_birth": None,
        "address": None,
        "postal_code": None,
        "email": None,
        "phone": None,
        "has_family_doctor": None,
        "family_doctor_name": None,
        "has_psychiatrist": None,
        "psychiatrist_name": None,
    },
    "current_concerns": {
        "primary_problems": None,
        "goals": None,
    },
    "past_psychiatric_history": {
        "previously_assessed": None,
        "assessment_details": None,
        "previous_diagnoses": None,
        "hospital_admission": None,
        "hospital_details": None,
        "self_harm_history": None,
        "self_harm_details": None,
        "previous_therapies": None,
        "previous_treatments": None,
    },
    "medications": {
        "has_current_medications": None,
        "current_medications": None,
        "has_past_medications": None,
        "past_medication_details": None,
    },
    "medical_history": {
        "has_medical_conditions": None,
        "medical_conditions": None,
        "has_surgical_history": None,
        "surgical_history": None,
        "height": None,
        "weight": None,
    },
    "medication_allergies": {
        "has_medication_allergies": None,
        "medication_allergies": None,
    },
    "family_psychiatric_history": {
        "has_family_psychiatric_history": None,
        "family_psychiatric_history_details": None,
    },
    "substance_use": {
        "alcohol_use": None,
        "nicotine_use": None,
        "marijuana_use": None,
        "stimulant_use": None,
        "opioid_use": None,
        "benzodiazepine_use": None,
        "hallucinogen_use": None,
        "other_substance_use": None,
    },
    "education_employment_legal_history": {
        "active_legal_cases": None,
        "legal_case_details": None,
        "highest_education": None,
        "employment_history": None,
    },
    "personal_history": {
        "early_childhood": None,
        "adolescence_early_adulthood": None,
        "significant_relationships": None,
        "future_goals": None,
    },
    "phq9": {
        "phq9_q1": None,
        "phq9_q2": None,
        "phq9_q3": None,
        "phq9_q4": None,
        "phq9_q5": None,
        "phq9_q6": None,
        "phq9_q7": None,
        "phq9_q8": None,
        "phq9_q9": None,
        "phq9_difficulty": None,
        "phq9_total": None,
    },
    "gad7": {
        "gad7_q1": None,
        "gad7_q2": None,
        "gad7_q3": None,
        "gad7_q4": None,
        "gad7_q5": None,
        "gad7_q6": None,
        "gad7_q7": None,
        "gad7_difficulty": None,
        "gad7_total": None,
    },
}

SECTION_TITLES = {
    "general_information": "General Information",
    "current_concerns": "Current Concerns",
    "past_psychiatric_history": "Past Psychiatric History",
    "medications": "Medications",
    "medical_history": "Medical History",
    "medication_allergies": "Medication Allergies",
    "family_psychiatric_history": "Family Psychiatric History",
    "substance_use": "Substance Use",
    "education_employment_legal_history": (
        "Education, Employment & Legal History"
    ),
    "personal_history": "Personal History",
    "phq9": "PHQ-9",
    "gad7": "GAD-7",
}

FIELD_LABELS = {
    "general_information": {
        "name": "Full Name",
        "gender": "Gender",
        "date_of_birth": "Date of Birth",
        "address": "Address",
        "postal_code": "Postal Code",
        "email": "Email",
        "phone": "Phone",
        "has_family_doctor": "Has Family Doctor",
        "family_doctor_name": "Family Doctor Name",
        "has_psychiatrist": "Has Psychiatrist",
        "psychiatrist_name": "Psychiatrist Name",
    },
    "current_concerns": {
        "primary_problems": "Primary Problems",
        "goals": "Appointment Goals",
    },
    "past_psychiatric_history": {
        "previously_assessed": "Previously Assessed",
        "assessment_details": "Assessment Details",
        "previous_diagnoses": "Previous Diagnoses",
        "hospital_admission": "Mental Health Hospital Admission",
        "hospital_details": "Hospital Admission Details",
        "self_harm_history": "History of Intentional Self-Harm",
        "self_harm_details": "History Details",
        "previous_therapies": "Previous Therapies",
        "previous_treatments": "Previous Treatments",
    },
    "medications": {
        "has_current_medications": "Currently Taking Prescribed Medications",
        "current_medications": "Current Medication Details",
        "has_past_medications": "Previously Tried Psychiatric Medications",
        "past_medication_details": "Past Medication Details",
    },
    "medical_history": {
        "has_medical_conditions": "Has Diagnosed Medical Conditions",
        "medical_conditions": "Medical Conditions",
        "has_surgical_history": "Has Surgical History",
        "surgical_history": "Surgical History",
        "height": "Height",
        "weight": "Weight",
    },
    "medication_allergies": {
        "has_medication_allergies": "Has Medication Allergies",
        "medication_allergies": "Medication Allergy Details",
    },
    "family_psychiatric_history": {
        "has_family_psychiatric_history": "Family Psychiatric History",
        "family_psychiatric_history_details": "Family History Details",
    },
    "substance_use": {
        "alcohol_use": "Alcohol",
        "nicotine_use": "Nicotine",
        "marijuana_use": "Marijuana / Cannabis",
        "stimulant_use": "Stimulants",
        "opioid_use": "Opiates / Narcotics",
        "benzodiazepine_use": "Benzodiazepines",
        "hallucinogen_use": "Hallucinogens",
        "other_substance_use": "Other Substances",
    },
    "education_employment_legal_history": {
        "active_legal_cases": "Active Legal Cases",
        "legal_case_details": "Legal Case Details",
        "highest_education": "Highest Education",
        "employment_history": "Employment History",
    },
    "personal_history": {
        "early_childhood": "Early Childhood",
        "adolescence_early_adulthood": "Adolescence & Early Adulthood",
        "significant_relationships": "Significant Relationships",
        "future_goals": "Future Goals",
    },
    "phq9": {
        "phq9_q1": "1. Little Interest or Pleasure",
        "phq9_q2": "2. Feeling Down, Depressed, or Hopeless",
        "phq9_q3": "3. Sleep Difficulties",
        "phq9_q4": "4. Tiredness or Low Energy",
        "phq9_q5": "5. Appetite Changes",
        "phq9_q6": "6. Negative Feelings About Yourself",
        "phq9_q7": "7. Difficulty Concentrating",
        "phq9_q8": "8. Slowing Down or Restlessness",
        "phq9_q9": "9. Thoughts of Being Better Off Absent or Self-Harm",
        "phq9_difficulty": "Functional Difficulty",
        "phq9_total": "PHQ-9 Total Score",
    },
    "gad7": {
        "gad7_q1": "1. Feeling Nervous, Anxious, or on Edge",
        "gad7_q2": "2. Unable to Stop or Control Worrying",
        "gad7_q3": "3. Worrying Too Much",
        "gad7_q4": "4. Trouble Relaxing",
        "gad7_q5": "5. Restlessness",
        "gad7_q6": "6. Easily Annoyed or Irritable",
        "gad7_q7": "7. Feeling Afraid Something Awful May Happen",
        "gad7_difficulty": "Functional Difficulty",
        "gad7_total": "GAD-7 Total Score",
    },
}

COMPUTED_FIELDS = {
    "phq9": {"phq9_total"},
    "gad7": {"gad7_total"},
}


# =========================================================
# STATE NORMALIZATION
# =========================================================

def normalize_state_schema(state):
    """
    Ensure every UI section exists and migrate the old medication schema.
    """
    if state is None:
        state = {}

    state.setdefault("messages", [])
    state.setdefault("current_section", None)
    state.setdefault("sections_completed", [])
    state.setdefault("safety_flag", False)
    state.setdefault("conversation_started", False)
    state.setdefault("latest_input_source", "typed text")
    state.setdefault("speech_to_text_active", False)

    legacy_current = state.get("current_medications", {})
    legacy_past = state.get("past_medications", {})

    if "medications" not in state:
        state["medications"] = {
            "has_current_medications": legacy_current.get(
                "has_current_medications"
            ),
            "current_medications": legacy_current.get(
                "current_medications"
            ),
            "has_past_medications": legacy_past.get(
                "has_past_medications"
            ),
            "past_medication_details": legacy_past.get(
                "past_medication_details"
            ),
        }

    if state.get("current_section") in {
        "current_medications",
        "past_medications",
    }:
        state["current_section"] = "medications"

    for section, defaults in SECTION_DEFAULTS.items():
        section_data = state.setdefault(section, {})

        for field, default_value in defaults.items():
            section_data.setdefault(field, default_value)

    return state


# =========================================================
# VOICE CONTROLS
# =========================================================

def get_latest_alice_message(history):
    """Return the latest assistant text currently displayed in the chat."""
    for message in reversed(history or []):
        if (
            isinstance(message, dict)
            and message.get("role") == "assistant"
        ):
            content = message.get("content")

            if isinstance(content, str):
                return content

    return None


def toggle_alice_voice(currently_enabled, history):
    """Toggle ALICE text-to-speech."""
    new_state = not currently_enabled

    if new_state:
        latest_message = get_latest_alice_message(history)
        audio_file = (
            text_to_speech(latest_message)
            if latest_message
            else None
        )

        return (
            new_state,
            gr.update(value="🔊 ALICE Voice: ON"),
            audio_file,
        )

    return (
        new_state,
        gr.update(value="🔇 ALICE Voice: OFF"),
        None,
    )


def toggle_patient_voice(currently_enabled, alice_state):
    """Toggle patient speech-to-text."""
    alice_state = normalize_state_schema(alice_state)
    new_state = not currently_enabled

    alice_state["speech_to_text_active"] = new_state
    alice_state["latest_input_source"] = "typed text"

    if new_state:
        return (
            new_state,
            alice_state,
            gr.update(value="🎤 Patient Voice: ON"),
            gr.update(visible=True, value=None),
            (
                "Patient Voice is on. Record an answer; "
                "it will be sent automatically."
            ),
        )

    return (
        new_state,
        alice_state,
        gr.update(value="🎤 Patient Voice: OFF"),
        gr.update(visible=False, value=None),
        "Patient Voice is off.",
    )


# =========================================================
# AVATARS
# =========================================================

def make_avatar(color, size=80):
    image = Image.new(
        "RGBA",
        (size, size),
        (0, 0, 0, 0),
    )

    draw = ImageDraw.Draw(image)
    draw.ellipse(
        [0, 0, size - 1, size - 1],
        fill=color,
    )

    path = os.path.join(
        tempfile.gettempdir(),
        f"avatar_{color[0]}_{color[1]}_{color[2]}.png",
    )

    image.save(path)
    return path


ALICE_AVATAR = "https://img.icons8.com/color/96/nurse-female.png"
USER_AVATAR = make_avatar((99, 115, 129))


# =========================================================
# INFORMATION PANEL
# =========================================================

def should_show_field(field, data):
    conditional_fields = {
        "family_doctor_name": ("has_family_doctor", False),
        "psychiatrist_name": ("has_psychiatrist", False),
        "assessment_details": ("previously_assessed", False),
        "hospital_details": ("hospital_admission", False),
        "self_harm_details": ("self_harm_history", False),
        "current_medications": ("has_current_medications", False),
        "past_medication_details": ("has_past_medications", False),
        "medical_conditions": ("has_medical_conditions", False),
        "surgical_history": ("has_surgical_history", False),
        "medication_allergies": ("has_medication_allergies", False),
        "family_psychiatric_history_details": (
            "has_family_psychiatric_history",
            False,
        ),
        "legal_case_details": ("active_legal_cases", False),
    }

    dependency = conditional_fields.get(field)

    if dependency is None:
        return True

    parent_field, hidden_value = dependency
    return data.get(parent_field) is not hidden_value


def format_display(alice_state):
    if not alice_state:
        return """
        <div class="empty-info">
            Select a section above to begin
        </div>
        """

    alice_state = normalize_state_schema(alice_state)
    section = alice_state.get("current_section")

    if not section:
        return """
        <div class="empty-info">
            Select a section above to begin
        </div>
        """

    data = alice_state.get(section, {})
    labels = FIELD_LABELS.get(section, {})
    title = SECTION_TITLES.get(
        section,
        section.replace("_", " ").title(),
    )

    visible_fields = {
        field: label
        for field, label in labels.items()
        if should_show_field(field, data)
        and not (
            field in COMPUTED_FIELDS.get(section, set())
            and data.get(field) is None
        )
    }

    progress_fields = [
        field
        for field in visible_fields
        if field not in COMPUTED_FIELDS.get(section, set())
    ]

    total = len(progress_fields)
    filled = sum(
        1
        for field in progress_fields
        if data.get(field) is not None
    )
    percentage = int((filled / total) * 100) if total else 0

    fields_html = ""

    for field, label in visible_fields.items():
        value = data.get(field)
        safe_label = html.escape(str(label))

        if value is None:
            value_html = '<div class="field-empty">—</div>'

        elif isinstance(value, bool):
            badge_class = "yes-badge" if value else "no-badge"
            badge_text = "Yes" if value else "No"
            value_html = (
                f'<span class="boolean-badge {badge_class}">'
                f"{badge_text}</span>"
            )

        elif field in {"phq9_total", "gad7_total"}:
            safe_value = html.escape(str(value))
            value_html = (
                '<div class="score-value">'
                f"{safe_value}"
                "</div>"
            )

        else:
            safe_value = html.escape(str(value))
            value_html = (
                f'<div class="field-value">{safe_value}</div>'
            )

        fields_html += f"""
        <div class="info-field">
            <div class="field-label">{safe_label}</div>
            {value_html}
        </div>
        """

    safe_title = html.escape(title)

    return f"""
    <div class="information-panel">
        <div class="information-header">
            <div>
                <div class="information-kicker">CURRENT SECTION</div>
                <div class="information-title">{safe_title}</div>
            </div>
            <div class="information-count">{filled}/{total}</div>
        </div>

        <div class="progress-track">
            <div
                class="progress-value"
                style="width:{percentage}%;">
            </div>
        </div>

        {fields_html}
    </div>
    """


# =========================================================
# CSS
# =========================================================

custom_css = """
@import url(
    'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap'
);

*, body, .gradio-container {
    font-family:
        'Inter',
        -apple-system,
        BlinkMacSystemFont,
        sans-serif !important;
}

body,
.gradio-container {
    background: #dce8f5 !important;
}

.gradio-container {
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 24px !important;
}

.alice-header {
    display: flex;
    align-items: center;
    gap: 16px;
    background: white;
    border: 1px solid #c5d8f0;
    border-radius: 16px;
    padding: 20px 26px;
    margin-bottom: 14px;
    box-shadow: 0 8px 24px rgba(30, 64, 175, 0.06);
}

.alice-header h1 {
    color: #1e293b !important;
    font-size: 23px !important;
    font-weight: 700 !important;
    margin: 0 !important;
    padding: 0 !important;
}

.alice-header p {
    color: #64748b !important;
    font-size: 13px !important;
    margin: 3px 0 0 !important;
}

.section-nav-label {
    margin: 2px 2px 8px;
    color: #475569;
    font-size: 12px;
    font-weight: 600;
    letter-spacing: 0.45px;
    text-transform: uppercase;
}

.section-nav-row {
    gap: 10px !important;
    margin-bottom: 10px;
}

.section-button button {
    width: 100% !important;
    min-height: 52px !important;
    padding: 11px 14px !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 11px !important;
    background: white !important;
    color: #1e293b !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    line-height: 1.25 !important;
    text-align: left !important;
    box-shadow: none !important;
    transition:
        transform 0.15s ease,
        background 0.15s ease,
        border-color 0.15s ease !important;
}

.section-button button:hover {
    background: #f8fbff !important;
    border-color: #93b4dc !important;
    transform: translateY(-1px);
}

.btn-general button { border-left: 4px solid #185FA5 !important; }
.btn-concerns button { border-left: 4px solid #0F6E56 !important; }
.btn-history button { border-left: 4px solid #7c3aed !important; }
.btn-medications button { border-left: 4px solid #d97706 !important; }
.btn-medical-history button { border-left: 4px solid #0284c7 !important; }
.btn-allergies button { border-left: 4px solid #dc2626 !important; }
.btn-family-history button { border-left: 4px solid #9333ea !important; }
.btn-substance-use button { border-left: 4px solid #0f766e !important; }
.btn-education button { border-left: 4px solid #4f46e5 !important; }
.btn-personal-history button { border-left: 4px solid #be123c !important; }
.btn-phq9 button { border-left: 4px solid #2563eb !important; }
.btn-gad7 button { border-left: 4px solid #0891b2 !important; }

.main-content-row {
    margin-top: 8px;
    align-items: stretch !important;
}

.chat-shell,
.info-shell {
    min-width: 0 !important;
}

.voice-card {
    min-height: 132px;
    background: white !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 12px !important;
    padding: 14px !important;
}

.voice-title {
    margin-bottom: 8px;
    color: #334155;
    font-size: 13px;
    font-weight: 600;
}

.voice-description {
    min-height: 34px;
    color: #64748b;
    font-size: 12px;
    line-height: 1.45;
}

.alice-voice-button button {
    border: 1px solid #93c5fd !important;
    background: #eff6ff !important;
    color: #1d4ed8 !important;
}

.patient-voice-button button {
    border: 1px solid #86efac !important;
    background: #f0fdf4 !important;
    color: #166534 !important;
}

#alice-audio-output {
    display: none !important;
}

.patient-recorder {
    margin-top: 8px !important;
}

.patient-status {
    color: #64748b;
    font-size: 12px;
}

.ai-disclosure {
    color: #64748b;
    font-size: 11px;
    margin-top: 4px;
}

.info-box {
    height: 100%;
    max-height: 760px;
    overflow-y: auto;
    background: #f8fbff !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 14px !important;
    padding: 18px !important;
}

.empty-info {
    padding: 36px 20px;
    color: #94a3b8;
    font-size: 13px;
    font-style: italic;
    text-align: center;
}

.information-panel {
    padding: 4px 8px;
}

.information-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 12px;
    padding-bottom: 12px;
    border-bottom: 2px solid #dbeafe;
}

.information-kicker {
    margin-bottom: 3px;
    color: #94a3b8;
    font-size: 9px;
    font-weight: 700;
    letter-spacing: 0.85px;
}

.information-title {
    color: #1e40af;
    font-size: 14px;
    font-weight: 700;
}

.information-count {
    color: #64748b;
    font-size: 11px;
    font-weight: 600;
}

.progress-track {
    height: 6px;
    margin-bottom: 16px;
    overflow: hidden;
    border-radius: 999px;
    background: #e2e8f0;
}

.progress-value {
    height: 6px;
    border-radius: 999px;
    background: linear-gradient(90deg, #185FA5, #3b82f6);
    transition: width 0.25s ease;
}

.info-field {
    padding: 11px 0;
    border-bottom: 1px solid #e8f0fb;
}

.field-label {
    margin-bottom: 5px;
    color: #94a3b8;
    font-size: 10px;
    letter-spacing: 0.55px;
    text-transform: uppercase;
}

.field-value {
    color: #1e293b;
    font-size: 14px;
    font-weight: 500;
    line-height: 1.5;
    overflow-wrap: anywhere;
    white-space: pre-wrap;
}

.field-empty {
    color: #cbd5e1;
    font-size: 13px;
    font-style: italic;
}

.boolean-badge {
    display: inline-block;
    padding: 3px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 600;
}

.yes-badge {
    color: #0F6E56;
    background: #e6f4ef;
}

.no-badge {
    color: #64748b;
    background: #eef2f7;
}

.score-value {
    display: inline-flex;
    min-width: 42px;
    height: 34px;
    align-items: center;
    justify-content: center;
    border-radius: 10px;
    background: #dbeafe;
    color: #1e40af;
    font-size: 17px;
    font-weight: 700;
}

@media (max-width: 900px) {
    .gradio-container {
        padding: 14px !important;
    }

    .section-nav-row {
        flex-wrap: wrap !important;
    }

    .section-button {
        min-width: calc(50% - 8px) !important;
    }

    .info-box {
        max-height: none;
    }
}

@media (max-width: 600px) {
    .gradio-container {
        padding: 10px !important;
    }

    .alice-header {
        padding: 16px 18px;
    }

    .section-button {
        min-width: 100% !important;
    }
}
"""


# =========================================================
# STATE AND CHAT CALLBACKS
# =========================================================

def get_fresh_state():
    state = copy.deepcopy(initial_state)
    return normalize_state_schema(state)


def start():
    """Initialize the UI. ALICE voice starts enabled."""
    state = get_fresh_state()

    welcome_text = (
        "Welcome! Please select a section above "
        "to begin your intake."
    )

    welcome_history = [{
        "role": "assistant",
        "content": f"👋 {welcome_text}",
    }]

    welcome_audio = text_to_speech(welcome_text)

    return (
        welcome_history,
        state,
        format_display(state),
        welcome_audio,
    )


def run_patient_message(
    message,
    displayed_user_message,
    history,
    alice_state,
    alice_voice_enabled,
    input_source,
    speech_to_text_active,
):
    """Shared logic for typed and spoken patient messages."""
    alice_state = normalize_state_schema(alice_state)
    history = history or []

    if alice_state.get("current_section") is None:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
        )

    if not message or not message.strip():
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
        )

    alice_state["latest_input_source"] = input_source
    alice_state["speech_to_text_active"] = speech_to_text_active

    alice_state["messages"] = (
        alice_state.get("messages", [])
        + [{
            "role": "user",
            "content": message.strip(),
        }]
    )

    alice_state = graph.invoke(alice_state)
    alice_state = normalize_state_schema(alice_state)
    reply = alice_state["messages"][-1]["content"]

    updated_history = history + [
        {
            "role": "user",
            "content": displayed_user_message,
        },
        {
            "role": "assistant",
            "content": reply,
        },
    ]

    reply_audio = (
        text_to_speech(reply)
        if alice_voice_enabled
        else None
    )

    return (
        updated_history,
        alice_state,
        format_display(alice_state),
        reply_audio,
    )


def chat(
    message,
    history,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """Submit a typed patient message to LangGraph."""
    clean_message = message.strip() if message else ""

    return run_patient_message(
        message=clean_message,
        displayed_user_message=clean_message,
        history=history,
        alice_state=alice_state,
        alice_voice_enabled=alice_voice_enabled,
        input_source="typed text",
        speech_to_text_active=patient_voice_enabled,
    )


def voice_chat(
    audio_file,
    history,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """Transcribe a recording and submit it immediately."""
    alice_state = normalize_state_schema(alice_state)
    history = history or []

    if not patient_voice_enabled:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "Patient Voice is off.",
        )

    if alice_state.get("current_section") is None:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "Please select an intake section before recording.",
        )

    transcript = transcribe_audio(audio_file)

    if not transcript:
        return (
            history,
            alice_state,
            format_display(alice_state),
            None,
            gr.update(value=None),
            "I could not understand the recording. Please try again.",
        )

    (
        updated_history,
        updated_state,
        updated_display,
        reply_audio,
    ) = run_patient_message(
        message=transcript,
        displayed_user_message="🎤 Voice response",
        history=history,
        alice_state=alice_state,
        alice_voice_enabled=alice_voice_enabled,
        input_source="speech-to-text",
        speech_to_text_active=True,
    )

    return (
        updated_history,
        updated_state,
        updated_display,
        reply_audio,
        gr.update(value=None),
        "Voice response sent automatically.",
    )


def switch_section(
    section,
    alice_state,
    alice_voice_enabled,
    patient_voice_enabled,
):
    """Open an intake section and ask its first pending question."""
    alice_state = normalize_state_schema(alice_state)

    alice_state["current_section"] = section
    alice_state["messages"] = []
    alice_state["latest_input_source"] = "typed text"
    alice_state["speech_to_text_active"] = patient_voice_enabled

    alice_state = graph.invoke(alice_state)
    alice_state = normalize_state_schema(alice_state)
    reply = alice_state["messages"][-1]["content"]

    section_history = [{
        "role": "assistant",
        "content": reply,
    }]

    reply_audio = (
        text_to_speech(reply)
        if alice_voice_enabled
        else None
    )

    return (
        section_history,
        alice_state,
        format_display(alice_state),
        reply_audio,
    )


def make_switch_handler(section_name):
    """Create a Gradio callback bound to one section."""
    def handler(state, alice_voice, patient_voice):
        return switch_section(
            section_name,
            state,
            alice_voice,
            patient_voice,
        )

    return handler


# =========================================================
# GRADIO INTERFACE
# =========================================================

with gr.Blocks(
    title="ALICE",
    css=custom_css,
    theme=gr.themes.Soft(),
) as demo:

    alice_state = gr.State(None)
    alice_voice_enabled = gr.State(True)
    patient_voice_enabled = gr.State(False)

    gr.HTML("""
        <div class="alice-header">
            <div style="
                width:52px;
                height:52px;
                border-radius:50%;
                background:#dbeafe;
                display:flex;
                align-items:center;
                justify-content:center;
                flex-shrink:0;
                font-size:26px;
            ">
                👩‍⚕️
            </div>

            <div>
                <h1>ALICE</h1>
                <p>Clinical Intake Assistant</p>
            </div>
        </div>
    """)

    gr.HTML(
        '<div class="section-nav-label">Intake Sections</div>'
    )

    section_buttons = {}

    with gr.Row(elem_classes="section-nav-row"):
        section_buttons["general_information"] = gr.Button(
            "General Information",
            elem_classes=["section-button", "btn-general"],
        )
        section_buttons["current_concerns"] = gr.Button(
            "Current Concerns",
            elem_classes=["section-button", "btn-concerns"],
        )
        section_buttons["past_psychiatric_history"] = gr.Button(
            "Past Psychiatric History",
            elem_classes=["section-button", "btn-history"],
        )
        section_buttons["medications"] = gr.Button(
            "Medications",
            elem_classes=["section-button", "btn-medications"],
        )

    with gr.Row(elem_classes="section-nav-row"):
        section_buttons["medical_history"] = gr.Button(
            "Medical History",
            elem_classes=["section-button", "btn-medical-history"],
        )
        section_buttons["medication_allergies"] = gr.Button(
            "Medication Allergies",
            elem_classes=["section-button", "btn-allergies"],
        )
        section_buttons["family_psychiatric_history"] = gr.Button(
            "Family Psychiatric History",
            elem_classes=["section-button", "btn-family-history"],
        )
        section_buttons["substance_use"] = gr.Button(
            "Substance Use",
            elem_classes=["section-button", "btn-substance-use"],
        )

    with gr.Row(elem_classes="section-nav-row"):
        section_buttons["education_employment_legal_history"] = gr.Button(
            "Education, Employment & Legal",
            elem_classes=["section-button", "btn-education"],
        )
        section_buttons["personal_history"] = gr.Button(
            "Personal History",
            elem_classes=["section-button", "btn-personal-history"],
        )
        section_buttons["phq9"] = gr.Button(
            "PHQ-9",
            elem_classes=["section-button", "btn-phq9"],
        )
        section_buttons["gad7"] = gr.Button(
            "GAD-7",
            elem_classes=["section-button", "btn-gad7"],
        )

    with gr.Row(elem_classes="main-content-row"):
        with gr.Column(
            scale=2,
            min_width=520,
            elem_classes="chat-shell",
        ):
            chatbot = gr.Chatbot(
                height=560,
                show_label=False,
                avatar_images=(
                    USER_AVATAR,
                    ALICE_AVATAR,
                ),
            )

            msg = gr.Textbox(
                placeholder="Type your message here...",
                show_label=False,
                container=False,
            )

            with gr.Row():
                with gr.Column(elem_classes="voice-card"):
                    gr.HTML("""
                        <div class="voice-title">ALICE Voice</div>
                        <div class="voice-description">
                            Reads ALICE's replies aloud automatically.
                        </div>
                    """)

                    alice_voice_button = gr.Button(
                        "🔊 ALICE Voice: ON",
                        elem_classes="alice-voice-button",
                    )

                with gr.Column(elem_classes="voice-card"):
                    gr.HTML("""
                        <div class="voice-title">Patient Voice</div>
                        <div class="voice-description">
                            Records and sends the patient's answer automatically.
                        </div>
                    """)

                    patient_voice_button = gr.Button(
                        "🎤 Patient Voice: OFF",
                        elem_classes="patient-voice-button",
                    )

            alice_audio_output = gr.Audio(
                autoplay=True,
                interactive=False,
                show_label=False,
                container=False,
                elem_id="alice-audio-output",
            )

            patient_audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Record your answer",
                visible=False,
                elem_classes="patient-recorder",
            )

            patient_status = gr.Markdown(
                "Patient Voice is off.",
                elem_classes="patient-status",
            )

            gr.Markdown(
                (
                    "ALICE's voice and patient transcription are AI-generated. "
                    "Patient recordings are transcribed and submitted automatically."
                ),
                elem_classes="ai-disclosure",
            )

        with gr.Column(
            scale=1,
            min_width=320,
            elem_classes="info-shell",
        ):
            info_display = gr.HTML(
                value=format_display(None),
                elem_classes="info-box",
                label="Collected Information",
            )

    alice_voice_button.click(
        toggle_alice_voice,
        inputs=[
            alice_voice_enabled,
            chatbot,
        ],
        outputs=[
            alice_voice_enabled,
            alice_voice_button,
            alice_audio_output,
        ],
    )

    patient_voice_button.click(
        toggle_patient_voice,
        inputs=[
            patient_voice_enabled,
            alice_state,
        ],
        outputs=[
            patient_voice_enabled,
            alice_state,
            patient_voice_button,
            patient_audio_input,
            patient_status,
        ],
    )

    patient_audio_input.stop_recording(
        voice_chat,
        inputs=[
            patient_audio_input,
            chatbot,
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
            patient_audio_input,
            patient_status,
        ],
    )

    msg.submit(
        chat,
        inputs=[
            msg,
            chatbot,
            alice_state,
            alice_voice_enabled,
            patient_voice_enabled,
        ],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    ).then(
        lambda: "",
        outputs=msg,
    )

    for section_name, button in section_buttons.items():
        button.click(
            make_switch_handler(section_name),
            inputs=[
                alice_state,
                alice_voice_enabled,
                patient_voice_enabled,
            ],
            outputs=[
                chatbot,
                alice_state,
                info_display,
                alice_audio_output,
            ],
        )

    demo.load(
        start,
        inputs=[],
        outputs=[
            chatbot,
            alice_state,
            info_display,
            alice_audio_output,
        ],
    )


demo.launch()


/Users/amir/Desktop/GitHub/Triage/Triage-Agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/v9/yz64p8z5579c6d6yx28446qh0000gn/T/ipykernel_22532/1522245827.py:1107: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/Users/amir/Desktop/GitHub/Triage/Triage-Agent/venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/amir/Desktop/GitHub/Triage/Triage-Agent/venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/amir/Desktop/GitHub/Triage/Triage-Agent/venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/amir/Desktop/GitHub/Triage/Triage-Agent/venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPRO